In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/studentInfo.csv')

In [3]:
df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [4]:
df.shape

(32593, 12)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code_module           32593 non-null  str  
 1   code_presentation     32593 non-null  str  
 2   id_student            32593 non-null  int64
 3   gender                32593 non-null  str  
 4   region                32593 non-null  str  
 5   highest_education     32593 non-null  str  
 6   imd_band              31482 non-null  str  
 7   age_band              32593 non-null  str  
 8   num_of_prev_attempts  32593 non-null  int64
 9   studied_credits       32593 non-null  int64
 10  disability            32593 non-null  str  
 11  final_result          32593 non-null  str  
dtypes: int64(3), str(9)
memory usage: 3.0 MB


In [6]:
df.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["imd_band"] = df["imd_band"].replace("None",pd.NA)

In [10]:
df.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

In [11]:
df.tail()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
32588,GGG,2014J,2640965,F,Wales,Lower Than A Level,10-20,0-35,0,30,N,Fail
32589,GGG,2014J,2645731,F,East Anglian Region,Lower Than A Level,40-50%,35-55,0,30,N,Distinction
32590,GGG,2014J,2648187,F,South Region,A Level or Equivalent,20-30%,0-35,0,30,Y,Pass
32591,GGG,2014J,2679821,F,South East Region,Lower Than A Level,90-100%,35-55,0,30,N,Withdrawn
32592,GGG,2014J,2684003,F,Yorkshire Region,HE Qualification,50-60%,35-55,0,30,N,Distinction


In [12]:
df["imd_band"] = df["imd_band"].fillna("Unknown")

In [13]:
df.isnull().sum()

code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

In [14]:
df.to_csv("studentInfo_cleaned.csv", index=False)

In [57]:
df= pd.read_csv("data/vle.csv")

In [58]:
df.head()

,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN
3,546888,AAA,2013J,url,NaN,NaN
4,547035,AAA,2013J,resource,NaN,NaN


In [59]:
df.shape

(6364, 6)

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_site            6364 non-null   int64  
 1   code_module        6364 non-null   str    
 2   code_presentation  6364 non-null   str    
 3   activity_type      6364 non-null   str    
 4   week_from          1121 non-null   float64
 5   week_to            1121 non-null   float64
dtypes: float64(2), int64(1), str(3)
memory usage: 298.4 KB


In [61]:
df.isnull().sum()

id_site                 0
code_module             0
code_presentation       0
activity_type           0
week_from            5243
week_to              5243
dtype: int64

In [63]:
import numpy as np
import pandas as pd



In [64]:
vle = pd.read_csv("data/vle.csv", na_values=["?", "", "NA"])
courses = pd.read_csv("data/courses.csv")
student_vle = pd.read_csv(
    "data/studentVle.csv",
    usecols=["id_site", "date"],
    dtype={"id_site": "int32", "date": "int16"},
)


In [65]:

# 1. Flag originally missing schedule rows as evergreen/course-wide resources
vle["is_evergreen"] = np.where(vle["week_from"].isna(), 1, 0).astype(np.int8)


In [66]:
# 2. Convert click day offsets into week numbers using NumPy
student_vle["click_week"] = np.floor(student_vle["date"] / 7).astype(np.int16)


In [67]:

# Compute 10th and 90th percentile active weeks per resource (id_site)
inferred_weeks = (
    student_vle.groupby("id_site")["click_week"]
    .agg(
        inferred_from=lambda s: int(np.percentile(s, 10)),
        inferred_to=lambda s: int(np.percentile(s, 90)),
    )
    .reset_index()
)


In [68]:

# 3. Merge inferred weeks & course duration into vle
vle = vle.merge(inferred_weeks, on="id_site", how="left")
vle = vle.merge(courses, on=["code_module", "code_presentation"], how="left")
max_weeks = np.ceil(vle["module_presentation_length"] / 7).astype(np.int16)


In [69]:

# 4. Fill missing week_from and week_to cleanly
vle["week_from"] = (
    vle["week_from"]
    .fillna(vle["inferred_from"])
    .fillna(0)
    .clip(lower=0)
    .astype(np.int16)
)

vle["week_to"] = (
    vle["week_to"]
    .fillna(vle["inferred_to"])
    .fillna(max_weeks)
    .clip(lower=vle["week_from"])  # Ensures week_to is never earlier than week_from
    .astype(np.int16)
)

# Drop temporary helper columns
vle.drop(columns=["inferred_from", "inferred_to", "module_presentation_length"], inplace=True)

In [70]:
vle.isnull().sum()

id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
is_evergreen         0
dtype: int64

In [71]:
vle.to_csv('vle_cleaned.csv', index=False)

In [56]:
df.to_csv('studentVle_cleaned.csv', index=False)

In [55]:
df.tail()

,code_module,code_presentation,id_student,id_site,date,sum_click
10655275,GGG,2014J,675811,896943,269,3
10655276,GGG,2014J,675578,896943,269,1
10655277,GGG,2014J,654064,896943,269,3
10655278,GGG,2014J,654064,896939,269,1
10655279,GGG,2014J,654064,896939,269,1


In [22]:
df["date"]=pd.to_datetime(df["date"], errors='coerce')

In [23]:
df.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,1970-01-01 00:00:00.000000019,10.0
1,AAA,2013J,1753,TMA,1970-01-01 00:00:00.000000054,20.0
2,AAA,2013J,1754,TMA,1970-01-01 00:00:00.000000117,20.0
3,AAA,2013J,1755,TMA,1970-01-01 00:00:00.000000166,20.0
4,AAA,2013J,1756,TMA,1970-01-01 00:00:00.000000215,30.0


In [24]:
df["date"] = pd.to_numeric(df["date"], errors="coerce")

In [25]:
df["date"] = df["date"].fillna(
    df.groupby(["code_module", "code_presentation"])["date"].transform("median")
)

In [26]:
df.isnull().sum()

code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

In [27]:
df.tail()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
201,GGG,2014J,37443,CMA,229,0.0
202,GGG,2014J,37435,TMA,61,0.0
203,GGG,2014J,37436,TMA,124,0.0
204,GGG,2014J,37437,TMA,173,0.0
205,GGG,2014J,37444,Exam,229,100.0


In [28]:
df.to_csv("assessments_cleaned.csv", index=False)

In [72]:
df = pd.read_csv("data/studentRegistration.csv")

In [73]:
df.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [74]:
df.shape

(32593, 5)

In [75]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  str    
 1   code_presentation    32593 non-null  str    
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 1.2 MB


In [76]:
df.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

In [78]:
import numpy as np
import pandas as pd



In [79]:
reg = pd.read_csv("data/studentRegistration.csv", na_values=["?", "", "NA"])
courses = pd.read_csv("data/courses.csv")


In [80]:

# Merge module_presentation_length so we know the exact end day of each course
reg = reg.merge(courses, on=["code_module", "code_presentation"], how="left")


In [81]:

# 1. Clean date_registration (~45 true nulls -> fill with module median)
reg["date_registration"] = pd.to_numeric(reg["date_registration"], errors="coerce")
reg["date_registration"] = reg["date_registration"].fillna(
    reg.groupby(["code_module", "code_presentation"])["date_registration"].transform("median")
).astype(np.int16)


In [82]:

# 2. Clean date_unregistration (~22,521 blanks -> students who stayed enrolled)
reg["date_unregistration"] = pd.to_numeric(reg["date_unregistration"], errors="coerce")


In [85]:

# Create a clean flag for your User Analytics dashboard
reg["is_withdrawn"] = np.where(reg["date_unregistration"].notna(), 1, 0).astype(np.int8)


In [86]:

# Compute exact active enrollment duration (using full course length if not withdrawn)
effective_end_day = reg["date_unregistration"].fillna(reg["module_presentation_length"])
reg["days_enrolled"] = np.maximum(0, effective_end_day - reg["date_registration"]).astype(np.int16)

In [87]:
reg.isnull().sum()

code_module                       0
code_presentation                 0
id_student                        0
date_registration                 0
date_unregistration           22521
module_presentation_length        0
is_withdrawn                      0
days_enrolled                     0
dtype: int64

In [88]:
courses = pd.read_csv("data/courses.csv")


In [96]:
reg = reg.merge(courses, on=["code_module", "code_presentation"], how="left")


In [93]:
reg = reg.drop(
    columns=["module_presentation_length", "module_presentation_length_x", "module_presentation_length_y"],
    errors="ignore",
)

In [98]:

reg["date_unregistration"] = pd.to_numeric(reg["date_unregistration"], errors="coerce")
reg["is_withdrawn"] = np.where(reg["date_unregistration"].notna(), 1, 0).astype(np.int8)

# Fill NaNs with the full module length (e.g., 268 or 270 days)
reg["date_unregistration"] = (
    reg["date_unregistration"]
    .fillna(reg["module_presentation_length"])
    .astype(np.int16)
)

In [97]:
reg.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration,is_withdrawn,days_enrolled,module_presentation_length
0,AAA,2013J,11391,-159,NaN,0,427,268
1,AAA,2013J,28400,-53,NaN,0,321,268
2,AAA,2013J,30268,-92,12.0,1,104,268
3,AAA,2013J,31604,-52,NaN,0,320,268
4,AAA,2013J,32885,-176,NaN,0,444,268


In [99]:
reg.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
date_registration             0
date_unregistration           0
is_withdrawn                  0
days_enrolled                 0
module_presentation_length    0
dtype: int64

In [100]:
reg.to_csv('studentRegistration_cleaned.csv', index=False)